In [ ]:
%pip install ultralytics


In [ ]:
import ultralytics
from ultralytics import YOLO
import torch.nn as nn
from torchvision.models import efficientnet_b0


In [3]:
class EfficientNetBackbone(nn.Module):
    def __init__(self):
        super(EfficientNetBackbone, self).__init__()
        efficientnet = efficientnet_b0(pretrained=True)
        self.features = efficientnet.features  # Use only feature extractor

    def forward(self, x):
        return self.features(x)

In [4]:
def replace_backbone_with_efficientnet(model):
    # Load the YOLO model
    model = YOLO(model)

    # Extract the original model architecture
    original_model = model.model  # Access the underlying PyTorch model

    # Replace the backbone
    original_model.backbone = EfficientNetBackbone()

    # Adjust the number of input channels if needed (optional)
    original_model.backbone.out_channels = 1280  # Output channels of EfficientNet-B0

    return model

In [ ]:
yolo_model_path = "yolo11n.pt"
modified_model = replace_backbone_with_efficientnet(yolo_model_path)


In [ ]:
print(modified_model.model)


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="PvEETTzxzsoQwlcklKW5")
project = rf.workspace("med-seg").project("planets-2_detection")
version = project.version(2)
dataset = version.download("yolov11")

In [ ]:
modified_model.train(data='/content/planets-2_detection-2/data.yaml', epochs=100)


In [ ]:
from IPython import display
from IPython.display import display, Image
Image(filename=f'/content/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
Image(filename=f'/content/runs/detect/train/results.png', width=600)

In [ ]:
import locale
locale.getpreferredencoding = lambda: 'UTF-8'

!yolo task=detect mode=val model=/content/runs/detect/train/weights/best.pt conf=0.25 source='/content/planets-2_detection-2/test/images' data='/content/planets-2_detection-2/data.yaml' hide_conf=True

In [ ]:
import locale
locale.getpreferredencoding = lambda: 'UTF-8'

!yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt conf=0.25 source='/content/planets-2_detection-2/test/images' hide_conf=True


In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:120]:
      display(Image(filename=image_path, width=600))
      print("\n")

In [ ]:
# Load the trained YOLOv11 model
model = YOLO("/content/runs/detect/train/weights/best.pt")  

# Evaluate the model on the test dataset
results = model.val(
    data="/content/planets-2_detection-2/data.yaml",  
    split="test",         # Evaluate on the test set
    imgsz=640,            # Image size for inference
    conf=0.25,            # Confidence threshold
    iou=0.6,              # IoU threshold for NMS
    device="0",           # Use GPU (set to "cpu" for CPU)
    save_json=True,       # Save results to JSON for further analysis
    save_conf=True,       # Save confidence scores in the JSON file
)

# Access metrics
metrics = results.box  # Access the Metrics object

# Print metrics
print(f"mAP50-95: {metrics.map}")       # mAP@0.5:0.95
print(f"mAP50: {metrics.map50}")         # mAP@0.5
print(f"mAP75: {metrics.map75}")         # mAP@0.75
print(f"Precision: {metrics.p.mean()}")  # Mean precision across all classes
print(f"Recall: {metrics.r.mean()}")     # Mean recall across all classes
print(f"F1 Score: {metrics.f1.mean()}")  # Mean F1 score across all classes
#print(f"Precision: {metrics.Precision}") # Precision
#print(f"Recall: {metrics.recall}")       # Recall
#print(f"IoU: {metrics.iou}")       # IoU
#print(f"IoU: {metrics.miou}")       # MIoU
#print(f"IoU: {metrics.f1}")       # f1-score
#print(f"IoU: {metrics.accuracy}")       # accuracy



# Print precision, recall, and F1 score for each class
for i, class_name in enumerate(model.names):
    print(f"Class: {class_name}")
    print(f"  Precision: {metrics.p[i]}")
    print(f"  Recall: {metrics.r[i]}")
    print(f"  F1 Score: {metrics.f1[i]}")



In [ ]:
import glob
import os
from PIL import Image
from IPython.display import display

# Path to the test folder
test_folder = "/content/planets-2_detection-2/test/images"

# Get all image file paths
image_paths = glob.glob(os.path.join(test_folder, "*.jpg")) 

# Run YOLO on all test images and hide confidence scores
results = model(image_paths, save=True, hide_conf=True)


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO, SAM
import os
from PIL import Image
from IPython.display import display


# Load the YOLO model
yolo_model = YOLO("/content/runs/detect/train/weights/best.pt") 

# Load the SAM model
sam_model = SAM("sam2_b.pt")

# Get class names
class_names = yolo_model.names  # Dictionary mapping class indices to class names

# Define input and output directories
input_folder = "/content/planets-2_detection-2/test/images"
output_folder = "/content/output_images"

os.makedirs(output_folder, exist_ok=True)

# Iterate over all images in the input folder
for image_name in os.listdir(input_folder):
    image_path = os.path.join(input_folder, image_name)

    # Run inference on the image
    results = yolo_model(image_path)  # Run inference

    # Load the image using OpenCV
    image = cv2.imread(image_path)

    # Process each YOLO result
    for result in results:
        boxes = result.boxes.xyxy  
        class_ids = result.boxes.cls.int().tolist()  # Get class IDs
        confidences = result.boxes.conf.tolist()  

        # Perform SAM segmentation
        sam_results = sam_model(result.orig_img, bboxes=boxes, verbose=False, save=True, device="cpu") 

        # Overlay segmentation masks
        for mask in sam_results[0].masks.data:
            mask = mask.cpu().numpy().astype(np.uint8)  
            colored_mask = np.zeros_like(image, dtype=np.uint8)
            colored_mask[mask == 1] = [255, 222, 33]  
            alpha = 0.5
            image = cv2.addWeighted(image, 1, colored_mask, alpha, 0)  

        # Draw bounding boxes and class labels
        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = map(int, box)  
            class_id = class_ids[i]
            #confidence = confidences[i]

            class_name = class_names.get(class_id, "Unknown") 
            #label = f"{class_name} {confidence:.2f}"
            label = f"{class_name}"

            # Draw bounding box
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)  


            #To make bg behind the text(label)
            text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]  
            text_w, text_h = text_size
            # Define the background rectangle coordinates
            bg_rect_start = (x1, y1 - text_h - 5)  
            bg_rect_end = (x1 + text_w + 10, y1)   
            # Draw blue background rectangle
            cv2.rectangle(image, bg_rect_start, bg_rect_end, (255, 0, 0), thickness=-1)  



            # Put class name and confidence on the image
            cv2.putText(image, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

        # Display SAM results without labels and confidence
        for sam_result in sam_results:
            sam_result.show(labels=False, conf=False)  # Hide labels & confidence

    output_path = os.path.join(output_folder, image_name)
    cv2.imwrite(output_path, image)   
    print(f"Image with bounding boxes, class names, and segmentation masks saved to: {output_path}")

for output_image_name in os.listdir(output_folder):
    output_image_path = os.path.join(output_folder, output_image_name)
    output_image = cv2.imread(output_image_path)
    display(Image.open(output_image_path))



In [ ]:
output_folder = "/content/output_images"
image_paths = glob.glob(os.path.join(output_folder, "*.jpg"))  # Change to ".png" if needed


# Display all processed images 
for image_path in image_paths:

    # Show the image
    display(Image.open(image_path))

In [ ]:
import os
import cv2
import numpy as np
import torch
from ultralytics import YOLO, SAM
from sklearn.metrics import accuracy_score

# Load YOLO and SAM models
yolo_model = YOLO("/content/runs/detect/train/weights/best.pt")  
sam_model = SAM("sam2_b.pt") 

# Define paths for the folder of images and ground truth labels
images_folder = "/content/planets-2_detection-2/test/images"
labels_folder = "/content/planets-2_detection-2/test/labels"

all_yolo_metrics = {"precision": [], "recall": [], "f1": [], "accuracy": []}
all_sam_metrics = {"iou": [], "dice": [], "accuracy": [], "sensitivity": [], "specificity": [], "miou": [], "map50": []}

# Function to compute IoU between two bounding boxes
def compute_iou(box1, box2):
    """
    Compute Intersection over Union (IoU) between two bounding boxes.
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union != 0 else 0

# Function to convert polygon coordinates to a binary mask
def polygon_to_mask(polygon, width, height):
    """
    Convert a polygon (list of x, y coordinates) to a binary mask.
    """
    mask = np.zeros((height, width), dtype=np.uint8)  
    polygon = np.array(polygon, dtype=np.int32).reshape((-1, 2)) 
    cv2.fillPoly(mask, [polygon], 1)  # Fill the polygon with 1s
    return mask

# Function to compute YOLO metrics
def compute_yolo_metrics(pred_boxes, pred_classes, gt_boxes, gt_classes, iou_threshold=0.5):
    """
    Compute YOLO metrics (precision, recall, F1-score) by matching predictions to ground truth.
    """
    tp = 0  # True positives
    fp = 0  # False positives
    fn = 0  # False negatives

    # Track which ground truth objects have been matched
    matched_gt = set()

    # For each predicted box, find the best matching ground truth box
    for i, pred_box in enumerate(pred_boxes):
        best_iou = 0
        best_match = -1
        for j, gt_box in enumerate(gt_boxes):
            if j in matched_gt:
                continue  # Skip already matched ground truth objects
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou and iou >= iou_threshold and pred_classes[i] == gt_classes[j]:
                best_iou = iou
                best_match = j
        if best_match != -1:
            tp += 1  
            matched_gt.add(best_match)
        else:
            fp += 1  

    fn = len(gt_boxes) - len(matched_gt)

    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

    return precision, recall, f1

# Function to compute YOLO accuracy
def compute_yolo_accuracy(pred_boxes, pred_classes, gt_boxes, gt_classes, iou_threshold=0.5):
    """
    Compute YOLO accuracy based on IoU and class matching.
    """
    tp = 0  # True positives
    fp = 0  # False positives
    fn = 0  # False negatives

    matched_preds = set() 
    for i, gt_box in enumerate(gt_boxes):
        best_iou = 0
        best_match = -1
        for j, pred_box in enumerate(pred_boxes):
            if j in matched_preds:
                continue  # Skip already matched predictions
            iou = compute_iou(gt_box, pred_box)
            if iou > best_iou and iou >= iou_threshold and pred_classes[j] == gt_classes[i]:
                best_iou = iou
                best_match = j
        if best_match != -1:
            tp += 1  # True positive
            matched_preds.add(best_match)
        else:
            fn += 1  # False negative

    fp = len(pred_boxes) - len(matched_preds)

    accuracy = tp / (tp + fp + fn) if (tp + fp + fn) != 0 else 0
    return accuracy

# Function to compute mIoU and mAP-50 for SAM
def compute_miou(pred_mask, gt_mask):
    """
    Compute mean Intersection over Union (mIoU) for segmentation.
    """
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    return intersection / union if union != 0 else 0

def compute_map50(pred_mask, gt_mask, iou_threshold=0.5):
    """
    Compute mean Average Precision at IoU threshold 0.50 (mAP-50) for segmentation.
    """
    # Flatten masks for pixel-wise comparison
    pred_mask_flat = pred_mask.flatten()
    gt_mask_flat = gt_mask.flatten()

    # Compute precision and recall
    precision = precision_score(gt_mask_flat, pred_mask_flat, zero_division=1)
    recall = recall_score(gt_mask_flat, pred_mask_flat, zero_division=1)

    # Compute AP (Average Precision)
    ap = precision if recall > 0 else 0
    return ap

for image_name in os.listdir(images_folder):
    if not image_name.endswith((".jpg", ".png")):
        continue  # Skip non-image files

    image_path = os.path.join(images_folder, image_name)
    gt_label_path = os.path.join(labels_folder, image_name.replace(".jpg", ".txt").replace(".png", ".txt"))

    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Image not found: {image_path}")
        continue

    # Get image dimensions
    height, width = image.shape[:2]

    yolo_results = yolo_model(image_path, conf=0.25)  
    pred_boxes = yolo_results[0].boxes.xyxy.cpu().numpy()  
    pred_classes = yolo_results[0].boxes.cls.cpu().numpy()  

    # Load ground truth labels (polygon format)
    gt_boxes, gt_classes, gt_mask = [], [], np.zeros((height, width), dtype=np.uint8)
    if os.path.exists(gt_label_path):
        with open(gt_label_path, "r") as f:
            for line in f.readlines():
                parts = list(map(float, line.strip().split()))
                if len(parts) < 6:  # Skip invalid lines (polygons must have at least 3 points)
                    print(f"Skipping invalid line: {line}")
                    continue
                class_id = int(parts[0])  # Class ID
                polygon = parts[1:]  # Polygon coordinates (x1, y1, x2, y2, ...)
                # Convert polygon coordinates from normalized to absolute values
                polygon = [(int(x * width), int(y * height)) for x, y in zip(polygon[::2], polygon[1::2])]
                # Add to ground truth mask
                gt_mask = np.maximum(gt_mask, polygon_to_mask(polygon, width, height))
                # Convert polygon to bounding box
                x_coords = [x for x, y in polygon]
                y_coords = [y for x, y in polygon]
                x1 = min(x_coords)
                y1 = min(y_coords)
                x2 = max(x_coords)
                y2 = max(y_coords)
                gt_boxes.append([x1, y1, x2, y2])
                gt_classes.append(class_id)
    else:
        print(f"Label file not found: {gt_label_path}")
        continue

    gt_boxes = np.array(gt_boxes)
    gt_classes = np.array(gt_classes)

    # Compute YOLO metrics
    if len(pred_classes) > 0 and len(gt_classes) > 0:
        precision, recall, f1 = compute_yolo_metrics(pred_boxes, pred_classes, gt_boxes, gt_classes)
        yolo_metrics = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "accuracy": compute_yolo_accuracy(pred_boxes, pred_classes, gt_boxes, gt_classes),
        }
    else:
        print(f"No ground truth labels found for {image_name}. YOLO metrics will be 0.")
        yolo_metrics = {"precision": 0, "recall": 0, "f1": 0, "accuracy": 0}

    sam_results = sam_model(image, bboxes=pred_boxes, device="cpu")

    # Combine all predicted masks
    pred_mask = np.zeros_like(gt_mask, dtype=np.uint8)
    for mask in sam_results[0].masks.data:
        mask_np = mask.cpu().numpy().astype(np.uint8)
        pred_mask = np.maximum(pred_mask, mask_np)  # Merge masks

    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    sam_metrics = {
        "iou": intersection / union if union != 0 else 0,
        "dice": (2 * intersection) / (pred_mask.sum() + gt_mask.sum()) if (pred_mask.sum() + gt_mask.sum()) != 0 else 0,
        "accuracy": accuracy_score(gt_mask.flatten(), pred_mask.flatten()),
        "sensitivity": recall_score(gt_mask.flatten(), pred_mask.flatten(), zero_division=1),
        "specificity": recall_score(1 - gt_mask.flatten(), 1 - pred_mask.flatten(), zero_division=1),
        "miou": compute_miou(pred_mask, gt_mask),
        "map50": compute_map50(pred_mask, gt_mask),
    }

    for key in yolo_metrics:
        all_yolo_metrics[key].append(yolo_metrics[key])
    for key in sam_metrics:
        all_sam_metrics[key].append(sam_metrics[key])

# Compute average metrics across all images
avg_yolo_metrics = {key: np.mean(values) for key, values in all_yolo_metrics.items()}
avg_sam_metrics = {key: np.mean(values) for key, values in all_sam_metrics.items()}

# Display average results
print("\n Average YOLO Object Detection Metrics:")
print(f" Precision: {avg_yolo_metrics['precision']:.4f}")
print(f" Recall: {avg_yolo_metrics['recall']:.4f}")
print(f" F1-score: {avg_yolo_metrics['f1']:.4f}")
print(f" Accuracy: {avg_yolo_metrics['accuracy']:.4f}")

print("\n Average SAM Segmentation Metrics:")
print(f"IoU: {avg_sam_metrics['iou']:.4f}")
print(f" Dice Score: {avg_sam_metrics['dice']:.4f}")
print(f" Accuracy: {avg_sam_metrics['accuracy']:.4f}")
print(f" Sensitivity: {avg_sam_metrics['sensitivity']:.4f}")
print(f" Specificity: {avg_sam_metrics['specificity']:.4f}")
print(f" mIoU: {avg_sam_metrics['miou']:.4f}")
print(f" mAP-50: {avg_sam_metrics['map50']:.4f}")